# FDO Connect

## FDO Nanopub Operations

This notebook provides examples for the operations that can be performed on FDOs as part of FDO Connect project to implement FAIR Digital Objects (FDOs) based on the existing Handle-based FDO One infrastructure and the technology of nanopublications.

In [6]:
from rdflib import RDF, URIRef

from nanopub import load_profile, NanopubConf
from nanopub.namespaces import NPX
from nanopub.fdo import FdoNanopub, validate_fdo_record, resolve_id
from pathlib import Path
from nanopub import Nanopub


In [7]:
conf = NanopubConf(
    add_prov_generated_time=False,
    add_pubinfo_generated_time=True,
    attribute_assertion_to_profile=True,
    attribute_publication_to_profile=True,
    profile=load_profile(),
    #use_test_server=True,
)

# FDO identifiers
specimen_doi = "10.3535/ZJX-6N5-A5C"       # Digital Specimen (DOI-based)
media_doi = "10.3535/SSL-ET3-MWX"          # Digital Media (DOI-based)
annotation_handle = "20.5000.1025/J7E-C1H-1X2"  # Annotation (Handle-based)


## Load, mark as example, validate, sign

For each FDO identifier:

1. Load the FDO record into a nanopub via `handle_to_nanopub`.
2. Mark it as an example nanopub (`npx:ExampleNanopub` in pubinfo).
3. Print the unsigned nanopub.
4. Validate the underlying FDO record.
5. Sign it and print the signed nanopub.

In [8]:
def mark_as_example(np):
    np.pubinfo.add((URIRef(str(np.metadata.np_uri)), RDF.type, NPX.ExampleNanopub))

def roundtrip_through_trig(np):
    """Mimic the CLI sign path: serialize unsigned nanopub to trig and reload,
    so rdflib's parser normalizes the in-memory graph before signing."""
    path = Path("/tmp") / "unsigned.trig"
    np.rdf.serialize(path, format="trig")
    return Nanopub(conf=conf, rdf=path)

def process(label, fdo_id):
    print(f"=== {label} ({fdo_id}) ===")
    np = FdoNanopub.handle_to_nanopub(fdo_id, conf=conf)
    mark_as_example(np)
    print("--- unsigned ---")
    print(np)
    print("--- validation ---")
    print(validate_fdo_record(resolve_id(fdo_id)))
    np = roundtrip_through_trig(np)
    np.sign()
    assert np.has_valid_signature and np.has_valid_trusty
    print("--- signed (valid) ---")
    print(np)
    return np

specimen_np = process("Digital Specimen", specimen_doi)
media_np = process("Digital Media", media_doi)
annotation_np = process("Annotation", annotation_handle)


=== Digital Specimen (10.3535/ZJX-6N5-A5C) ===
--- unsigned ---
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix ns1: <https://hdl.handle.net/> .
@prefix ns2: <https://hdl.handle.net/10320/> .
@prefix ns3: <https://w3id.org/fdof/ontology#> .
@prefix orcid: <https://orcid.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://purl.org/nanopub/temp/np/provenance> {
    <http://purl.org/nanopub/temp/np/assertion> prov:wasAttributedTo orcid:0009-0009-0118-9195 .
}

<http://purl.org/nanopub/temp/np/pubinfo> {
    <http://purl.org/nanopub/temp/np/> a npx:ExampleNanopub ;
        rdfs:label "FAIR Digital Object: Rumex alpinus L." ;
        npx:introduces <https://hdl.handle.net/10.3535/ZJX-6N5-A5C> ;
        prov:generatedAtTime "2026-04-16T10:57:04.144639"^^xsd:dateTime ;
        pro

## Save to local files

Saves the three signed nanopubs to `signed_fdos/` so they can be inspected or published via the CLI (`np publish <file>.trig`).

In [9]:
out_dir = Path("signed_fdos")
out_dir.mkdir(exist_ok=True)

for label, np in [("specimen", specimen_np), ("media", media_np), ("annotation", annotation_np)]:
    path = out_dir / f"{label}.trig"
    np.rdf.serialize(path, format="trig")
    print(f"Saved {label}: {path}  ({np.source_uri})")

Saved specimen: signed_fdos/specimen.trig  (https://w3id.org/np/RAUKLGjEM3jZcfFyYzIshnVJqu0k10Lq41UZaSYSytFms)
Saved media: signed_fdos/media.trig  (https://w3id.org/np/RAlEAoNSBXWtiP8NpuCTqUaw16hqn6fiaiKWRzTUpJd58)
Saved annotation: signed_fdos/annotation.trig  (https://w3id.org/np/RAnwL88bA1ySwv3Dpj0mm0LRZ8GP5UKVKjYmOpxaVECe8)


## Publish

Publishes the three signed nanopubs to the configured server.

In [10]:
from nanopub.sign_utils import publish_graph

SERVER = "https://registry.petapico.org/np/"

for np in (specimen_np, media_np, annotation_np):
    publish_graph(np.rdf, use_server=SERVER)
    np.published = True
    print(f"Published {np.source_uri} to {SERVER}")


HTTPError: 400 Client Error: Error processing nanopub: Nanopublication not supported: https://w3id.org/np/RAUKLGjEM3jZcfFyYzIshnVJqu0k10Lq41UZaSYSytFms for url: https://registry.petapico.org/np/

In [ ]:
import sys, nanopub
print(sys.executable)
print(nanopub.__file__)
print(nanopub.__version__)
